# **Scraping Planos Super Telecom**

_Página do provedor: https://superitelecom.com.br/_

In [1]:
# Bibliotecas
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [2]:
# Obter lista de planos
texto_html = open("../../data/scraped/super.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
planos = soup.find_all("div", role="article")
print(f"{len(planos)} cartões de planos capturados.")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/scraped/super.html'

In [ ]:
# Extrair informações dos planos
def raspar_plano(plano):
    velocidade = plano.find("span", class_="text-6xl").text
    preco = plano.find("span", class_="text-2xl").text
    pre_sva = plano.find_all("span", class_="text-[9px]")
    lista_sva = [i.text for i in pre_sva]
    svas = ", ".join(lista_sva)

    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

dados = []

for i in planos:
    dados_plano = raspar_plano(i)
    dados.append(dados_plano)

df = pd.DataFrame(dados)
df

## **2. Tratamento**

In [ ]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Super"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"]

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

## **3. Carga**

In [ ]:
# Salvando
provedor = "SUPER.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")